# Use environment variables to select execution parameters for assessing Data Quality

In [ ]:
import os

os.chdir("../../../")

In [ ]:
os.environ['DQ_MODE'] = 'incremental'
os.environ['DQ_DATASOURCE'] = 'country_sales'
os.environ['DQ_LAYER'] = 'gold'
os.environ['DQ_LIMIT'] = '100'

- ### Initialize the Spark connection
- ### Read data based on the selected parameters

In [ ]:
from data_quality.gx.plugins.connectors import SparkConnector

connector = SparkConnector()
connector.connect()
dtlk_df = connector.read_data()

In [ ]:
datasource = connector.datasource
dtlk_df_layer = connector.layer

dq_suite = f"{dtlk_df_layer}_{datasource}_suite"
dq_asset = f"{dtlk_df_layer}_{datasource}_asset"

## Initialize Great Expectations:

### Setup GX context:

 - _Define GX Data Source:_
   - A Data Source provides a standard API for accessing and interacting with data from a wide variety of source systems.
 - _Define GX Data Asset:_
    - A Data Asset is a collection of records within a Data Source which is usually named based on the underlying data system and sliced specification.
 - Define Batch:
    - A Batch is a selection of records from a Data Asset.
 - Define Expectation Suite:
   - An Expectation Suite is a collection of verifiable assertions about data.
 - Define GX validator:
    - A Validator is the object responsible for running an Expectation Suite against data.

In [ ]:
import great_expectations as gx

context = gx.get_context(project_root_dir="./data_quality")

context.add_or_update_expectation_suite(expectation_suite_name=dq_suite)

dataframe_datasource = context.sources.add_or_update_spark(
    name=f"{dtlk_df_layer}_{datasource}",
)
dq_asset = dataframe_datasource.add_dataframe_asset(
    name=dq_asset,
    dataframe=dtlk_df,
)
dq_request = dq_asset.build_batch_request()

dq_validator = context.get_validator(
    batch_request=dq_request,
    expectation_suite_name=dq_suite,
)

dq_validator.interactive_evaluation = False

### Define Completeness checks:

> Availability of required data attributes:
>  - There are no gaps in data structure (all fields are populated)
> - Checks for data gaps, NULL distribution, column shifts and etc

#### GX methods:

 - _expect_column_to_exist()_ - Expect the specified column to exist.

 - _expect_column_values_to_not_be_null()_ - Expect the column values to not be null.

 - _expect_table_row_count_to_be_between()_ - Expect the number of rows to be between two values. Possible to add __max__ value as well.

 - _expect_table_column_count_to_equal()_ - Expect the number of columns in a table to equal a value. Method counts not only high-level but also every nested column in _STRUCT_ attributes.

In [ ]:
from data_quality.gx.plugins.metadata import METADATA

metadata = {"dimension": "Completeness", "Layer": dtlk_df_layer}

for attribute, attribute_info in METADATA[dtlk_df_layer][datasource]['model'].items():
    dq_validator.expect_column_to_exist(attribute, meta=metadata)
    if not attribute_info['nullable']:
        dq_validator.expect_column_values_to_not_be_null(attribute, meta=metadata)

dq_validator.expect_table_row_count_to_be_between(
    min_value=2,
    meta=metadata
)

dq_validator.expect_table_column_count_to_equal(
    value=4,
    meta=metadata
)

### Define Timeliness checks:

> The currency of content representation as well as whether the data is available/can be used when needed:
> - Data is available upon request and when required.
> - Measures are established to track and when possible, optimize data load times, report generation times, interface response times.

#### GX methods:
 - _expect_column_max_to_be_between()_ - Expect the column maximum to be between a minimum value and a maximum value.

In [ ]:
import datetime

metadata = {"dimension": "Timeliness", "Layer": dtlk_df_layer}

now = datetime.datetime.now()
start_of_year = datetime.datetime(now.year, 1, 1)
days_back = (now - start_of_year).days + 1
min_date = now - datetime.timedelta(days=days_back)

dq_validator.expect_column_max_to_be_between(
    column='gold_generated_timestamp',
    min_value=min_date,
    meta=metadata
)

### Define Validity checks:

> Alignment of data content with required standards:
> - Values conform to pre-defined ranges.
> - No format mismatches.
> - No issues with casting data types when loading.

#### GX methods:
- _expect_column_values_to_match_regex()_ - Expect the column entries to be strings that match a given regular expression.
- _expect_column_values_to_be_of_type()_ - Expect a column to contain values of a specified data type.
- _expect_column_values_to_be_between()_ - Expect the column value to be between a minimum value and a maximum value.
- _expect_column_values_to_be_json_parseable()_ - Expect the column entries to be data written in Json (JavaScript Object Notation).

In [ ]:
metadata = {"dimension": "Validity", "Layer": dtlk_df_layer}

dq_validator.expect_column_values_to_match_regex(
    column='gold_generated_timestamp',
    regex='\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])\s([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]\.\d+',
    meta=metadata
)

positive_columns = ["total_revenue", "avg_profit_margin_pct"]
for column in positive_columns:
    dq_validator.expect_column_values_to_be_between(
        column=column,
        min_value=0.01,
        strict_min=True,
        meta=metadata
    )

for attribute, attribute_info in METADATA[dtlk_df_layer][datasource]['model'].items():
    dq_validator.expect_column_values_to_be_of_type(
        column=attribute,
        type_=attribute_info['data_type'],
        meta=metadata
    )

### Define Integrity checks:

> How well the data complies with the required formats/definitions:
> - The structure of relationships within the data is correctly maintained
> - Keys and relationships are properly established and preserved, enabling complex queries across multiple datasets

#### GX methods:
- _expect_column_values_to_be_in_set()_ - Ensures that each value in a column belongs to a predefined set of valid value(s)

In [ ]:
metadata = {"dimension": "Integrity", "Layer": dtlk_df_layer}

# RefIntegrity
value_type = connector.spark.read.format("delta").load(f"{connector.datasource_map['reference']['geography']['path_prefix']}")
values =  value_type.select("country_name").distinct().rdd.flatMap(lambda x: x).collect()
dq_validator.expect_column_values_to_be_in_set(
    "customer_country_name",
    value_set=values,
    meta=metadata
)

### Define Uniqueness checks:

> Focus on garanting the uniquenness of values or combinations of values within data entity:
> - Uniqueness constraint violations of primary key(s)
> - Uniqueness for columns (or their combinations) that contain business-critical or domain-specific logic

#### GX methods:
- _expect_compound_columns_to_be_unique()_ - Expect the compound columns to be unique.
- _expect_column_values_to_be_unique()_ - Expect each column value to be unique. This expectation detects duplicates. All duplicated values are counted as exceptions.

In [ ]:
metadata = {"dimension": "Uniqueness", "Layer": dtlk_df_layer}

dq_validator.expect_compound_columns_to_be_unique(
    column_list=['customer_country_name','total_revenue','avg_profit_margin_pct','gold_generated_timestamp'],
    meta=metadata
)

### Perform checks and Calculate metrics

- A GX Checkpoint serves as the main mechanism for validating data.
- Checkpoints offer a practical abstraction that groups the Validation of one or more data Batches against an Expectation Suite.

In [ ]:
dq_validator.save_expectation_suite(discard_failed_expectations=False)


checkpoint = context.add_or_update_checkpoint(
    name=f"{dtlk_df_layer}_{datasource}_checkpoint",
    run_name_template=f"%Y%m%d-%H%M%S-{dtlk_df_layer}-{datasource}",
    validations=[
        {
            "batch_request":dq_request,
            "expectation_suite_name":dq_suite,
        },
    ],
    action_list=[
        {
            "name": "store_validation_result",
            "action": {"class_name": "StoreValidationResultAction"}
        },
    ],
)

context.add_or_update_checkpoint(checkpoint=checkpoint)

checkpoint_results = checkpoint.run()
context.build_data_docs()
# context.open_data_docs()

## Generate Custom Data Quality Report

In [ ]:
connector.generate_report(checkpoint_results, connector.mode)
connector.show_report(connector.layer, connector.datasource, detailed_report=True)
# connector.save_report()